# <u>Ridge, Lasso & Elastic Net Regression</u>

### Prerequisites:
* <a href="../2.Multiple%20linear%20Regression/Multiple%20linear%20Regression.ipynb">Check out the notebook on Multiple linear Regression</a>

## Topics

* [Review OLS derivation for Multiple linear Regression](#review)

* [0. Problem with unregularized multiple linear regression](#problem)
    
* [1. Ridge Regression](#ridge)
    * [1.1 Motivation](#motivation1)
    * [1.2 Derivation](#derive1)
    * [1.3 Implementation](#implement1)

* [2. Lasso Regression](#lasso)
    * [2.1 Motivation](#motivation2)
    * [2.2 Derivation](#derive2)
    * [2.3 Implementation](#implement2)

* [3. Lasso vs Ridge Regression](#lasso)
    * [3.1 L1 Penalty vs L2 Penalty](#L1L2)
    * [3.2 Sparsity vs Shrinkage](#sparsevsshrink)
    * [3.3 Visualizations](#vision)

* [4. Elastic Net Regression](#lasso)
    * [4.1 Motivation](#motivation1)
    * [4.2 Formula](#derive1)
    * [4.3 Implementation](#implement1)

* [5. Robust linear Regression](#robust)

<a class="anchor" id="review"></a>
## Review OLS derivation for Multiple linear Regression

In multiple linear regression, we model the relationship between a response variable $y$ and $p$ predictors $X \in \mathbb{R}^{n \times (p+1)}$ using 

$$
y=X\theta + \varepsilon \Leftrightarrow 
\begin{pmatrix}
y^{(1)} \\
y^{(2)} \\
\vdots \\
y^{(n)}
\end{pmatrix} =
\begin{pmatrix}
1 & x_1^{(1)} & \ldots & x_p^{(1)} \\
1 & x_1^{(2)} & \ldots & x_p^{(2)} \\
\vdots & \vdots & \ddots & \vdots \\
1 & x_1^{(n)} & \ldots & x_p^{(n)}
\end{pmatrix}
\begin{pmatrix}
\theta_0 \\
\theta_1 \\
\vdots \\
\theta_p
\end{pmatrix} +
\begin{pmatrix}
\varepsilon^{(1)} \\
\varepsilon^{(2)} \\
\vdots \\
\varepsilon^{(n)}
\end{pmatrix}
$$

We then estimate $\theta$ with $\hat{\theta}$ to make predictions for new data 
$$
\hat{y}=X \hat{\theta}
$$

We can derive the ordinary least squares (OLS) estimator in three ways:
1. Analytical solution (set derivative to 0 and solve)
2. Bayesian statistics
3. Optimization (Gradient Descent)



<div style="display:flex; gap:20px;">

<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--color:white;-->
width:50%;
">


<h5 style="text-align:center;"><u>Analytical solution</u></h5>

**Goal:** Find parameter vector $\hat{\theta} \in \mathbb{R}^{p+1}$ by finding parameter vector $\theta \in \mathbb{R}^{p+1}$
that minizes sum of squared errors.

$$
\begin{align*}
\hat{\theta} 
&= \arg \min_{\theta \in \mathbb{R}^{p+1}} \lVert y - X\theta \rVert_2^2 \\
&= \arg \min_{\theta \in \mathbb{R}^{p+1}} (y-X\theta)^\top(y-X\theta) \\
&= \frac{\partial}{\partial \theta}  (y - X\theta)^\top (y - X\theta) \\
&= \frac{\partial}{\partial \theta} \left(y^\top y  - 2\theta^\top X^\top y + \theta^\top X^\top X\theta \right) \\
&=  - 2X^\top y + 2 X^\top X\theta 
\end{align*}
$$

Setting the derivative to 0 yields

$$
\begin{align*}
- 2X^\top y + 2 X^\top X\theta &= 0 \\ \Leftrightarrow
2 X^\top X\theta &= 2X^\top y \\ \Leftrightarrow
X^\top X\theta &= X^\top y \\ \Leftrightarrow
(X^\top X)^{-1}X^\top X\theta &= (X^\top X)^{-1}X^\top y \\ \Leftrightarrow
\theta &= (X^\top X)^{-1} X^\top y  \\
\Rightarrow \hat{\theta} &= (X^\top X)^{-1} X^\top y
\end{align*}
$$



</div>


<!--  -->
<div style="
padding:16px;
border-radius:8px;
<!--color:white;-->
width:50%;
">

<h5 style="text-align:center;"><u>Bayesian statistics</u></h5>

**Goal:** Find parameter vector $\hat{\theta} \in \mathbb{R}^{p+1}$ by finding parameter vector $\theta \in \mathbb{R}^{p+1}$
that maximizes the Posterior distribution $p(\theta \mid y,X)$ over $\theta$.

In the notebook Multiple linear Regression in section <a href="../2.Multiple%20linear%20Regression/Multiple%20linear%20Regression.ipynb#min">Core Minimization problem (OLS)</a> we derive the OLS solution using only the (log) likelihood $p(y \mid X ,\theta)$ but the true general setup includes a prior $p(\theta)$ quantifying our belief of how the true $\theta$ looks like before seeing any data. 

$$
\text{Likelihood: }y \mid X,\theta \sim \mathcal{N}(X\theta, \sigma^2\mathrm{I}) \text{ since }\varepsilon \sim \mathcal{N}(0, \sigma^2\mathrm{I}) \\

\text{Prior: } \theta \sim \mathcal{U}(-\infty,\infty)
$$

$$
\begin{align*}
\hat{\theta} 
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} p(\theta \mid y,X) \\
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} \frac{p(y \mid X,\theta)p(\theta)}{p(\mathcal{D})} 
\end{align*}
$$ 

- $\mathcal{D}=\left\{ \left( (x_1^{(i)},x_2^{(i)}, \ldots, x_p^{(i)}),y^{(i)} \right)\right\}_{i=1}^n$ 
- $p(\mathcal{D})$ not dependent on $\theta$

$$
\begin{align*}
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} p(y \mid X,\theta)p(\theta)
\end{align*}
$$ 

- $p(\theta) \propto 1$

$$
\begin{align*}
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} p(y \mid X,\theta)
\end{align*}
$$ 

- Taking the logarith of $p(y \mid X,\theta)$ make calculations more easily and does not change results 

$$
\begin{align*}
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} \log p(y \mid X,\theta) \\
&= \arg \max_{\theta \in \mathbb{R}^{p+1}} -\frac{n}{2} \log (2\pi\sigma^2) - \frac{1}{2\sigma^2} (y-X\theta)^\top(y-X\theta)
\end{align*}
$$ 

- Since $-\frac{n}{2} \log (2\pi\sigma^2)$ and $\frac{1}{2\sigma^2}$ are independent from $\theta$ and multiplying with $-1$ means we no longer maximize but minimize we get

$$
\begin{align*}
&= \arg \min_{\theta \in \mathbb{R}^{p+1}}  (y-X\theta)^\top(y-X\theta) \\
&= \vdots \\
\Rightarrow \hat{\theta} &= (X^\top X)^{-1} X^\top y
\end{align*}
$$ 

</div>

</div>







<a class="anchor" id="problem"></a>
## 0. Problems with unregularized multiple linear regression

$$
\hat{\theta}=(X^\top X)^{-1} X^\top y
$$

**This estimator works well under ideal conditions, but in practice unregularized multiple linear regression suffers from several serious problems.**

$$
\begin{align*}
\text{Problems } &=
\left\{ 
\begin{array}{rcl}
\text{Case 1:} & \text{More features than observations } & p > n  \\ 
\text{Case 2:} & \text{Multicollinearity of features } & x_2 \approx 2 \cdot x_1 + \text{small noise}
\end{array}
\right\}
\end{align*}
$$

$$
\begin{align*}
\text{Consequence: } 
& - X^\top X \text{ becomes singular or nearly singular (rank}(X^\top X) < p\text{)} \\
& -\text{Estimated coefficients become very large and unstable} \\
& -\text{Small changes in data cause large changes in estimated coefficients}
\end{align*}
$$

Note:
- Perfect multicollinearity = linear dependence
- Weak/Moderate multicollinearity = near linear dependence

| Case              | $X^TX$          | OLS                   |
| ----------------- | --------------- | --------------------- |
| Linear dependence | singular        | no unique solution    |
| Multicollinearity | nearly singular | unstable coefficients |


**This leads to a high-variance estimator and poor generalization performance.**